# Captures 3D — Video to 3D Gaussian Splat Pipeline\n\nThis notebook converts your **video** or **images** into a navigable **3D Gaussian Splat** scene.\n\n**Pipeline:** Video → Frame Extraction → COLMAP (camera poses) → Nerfstudio splatfacto → `.splat` file\n\n**Output:** A `.splat` file you can view in the [Captures 3D Viewer](https://captures-3d-ognvwkqc.devinapps.com)\n\n---\n\n**Requirements:**\n- Google Colab with **GPU runtime** (free T4 works)\n- A video file (30-60 seconds, slow pan, good lighting)\n\n**⚠️ Before starting:** Go to `Runtime → Change runtime type → GPU (T4)`

## Step 1: Check GPU & Install Dependencies\n\nThis installs COLMAP, Nerfstudio, and all required packages. Takes ~3-5 minutes.

In [ ]:
# Check GPU availability
import subprocess
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if result.returncode != 0:
    print("❌ No GPU detected!")
    print("Go to: Runtime → Change runtime type → GPU (T4)")
    raise RuntimeError("GPU required. Change runtime type to GPU.")
else:
    # Extract GPU name
    for line in result.stdout.split("\n"):
        if "Tesla" in line or "A100" in line or "V100" in line or "T4" in line or "RTX" in line:
            print(f"✅ GPU detected: {line.strip()}")
            break
    else:
        print("✅ GPU detected")
    print(result.stdout)

In [ ]:
# Install COLMAP
!apt-get update -qq && apt-get install -y -qq colmap ffmpeg > /dev/null 2>&1
!colmap --help > /dev/null 2>&1 && echo "✅ COLMAP installed" || echo "❌ COLMAP failed"
!ffmpeg -version 2>/dev/null | head -1 && echo "✅ ffmpeg installed"

In [ ]:
# Install Nerfstudio (takes ~3-5 min)
!pip install nerfstudio 2>&1 | tail -5
!ns-train --help > /dev/null 2>&1 && echo "✅ Nerfstudio installed" || echo "❌ Nerfstudio install failed"

## Step 2: Upload Your Video\n\nUpload a video file. **Tips for best results:**\n- 30-60 seconds long\n- Slow, steady camera movement (no fast panning)\n- Good, consistent lighting\n- Avoid motion blur and reflective surfaces

In [ ]:
from google.colab import files
import os

print("📁 Select your video file...")
uploaded = files.upload()

VIDEO_FILE = list(uploaded.keys())[0]
VIDEO_PATH = f"/content/{VIDEO_FILE}"
print(f"✅ Uploaded: {VIDEO_FILE} ({os.path.getsize(VIDEO_PATH) / 1024 / 1024:.1f} MB)")

## Step 3: Configure Pipeline Settings\n\nAdjust these settings based on your needs:

In [ ]:
#@title Pipeline Settings { run: "auto" }

#@markdown ### Frame Extraction
TARGET_FPS = 2  #@param {type:"slider", min:1, max:5, step:1}
#@markdown How many frames per second to extract. 2 is recommended.

#@markdown ### Training Quality
QUALITY = "standard"  #@param ["quick", "standard", "high"]
#@markdown - **quick**: ~5 min, ~60% quality (7K iterations)
#@markdown - **standard**: ~15 min, ~80% quality (15K iterations)
#@markdown - **high**: ~30 min, ~90% quality (30K iterations)

# Set iterations based on quality
ITERATIONS = {"quick": 7000, "standard": 15000, "high": 30000}[QUALITY]

# Directory setup
WORK_DIR = "/content/captures3d"
FRAMES_DIR = f"{WORK_DIR}/frames"
PROCESSED_DIR = f"{WORK_DIR}/processed"
MODEL_DIR = f"{WORK_DIR}/model"
OUTPUT_DIR = f"{WORK_DIR}/output"

os.makedirs(FRAMES_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"⚙️ Settings: {TARGET_FPS} fps, {QUALITY} quality ({ITERATIONS} iterations)")
print(f"📂 Working directory: {WORK_DIR}")

## Step 4: Extract Frames from Video\n\nExtracts frames at your chosen FPS using ffmpeg.

In [ ]:
import time

print(f"🎬 Extracting frames at {TARGET_FPS} fps...")
start = time.time()

!ffmpeg -i "{VIDEO_PATH}" -vf "fps={TARGET_FPS}" -q:v 2 "{FRAMES_DIR}/frame_%05d.jpg" -y 2>&1 | tail -3

num_frames = len([f for f in os.listdir(FRAMES_DIR) if f.endswith(".jpg")])
elapsed = time.time() - start
print(f"✅ Extracted {num_frames} frames in {elapsed:.1f}s")

## Step 5: Compute Camera Poses (COLMAP)\n\nUses Nerfstudio's `ns-process-data` to run COLMAP and estimate camera positions. This takes ~2-10 min depending on frame count.

In [ ]:
print("📐 Running COLMAP to compute camera poses...")
print("   (This may take 2-10 minutes depending on number of frames)")
start = time.time()

!ns-process-data images \
    --data "{FRAMES_DIR}" \
    --output-dir "{PROCESSED_DIR}" \
    2>&1 | tail -10

elapsed = time.time() - start

# Verify output
transforms_path = f"{PROCESSED_DIR}/transforms.json"
if os.path.exists(transforms_path):
    import json
    with open(transforms_path) as f:
        transforms = json.load(f)
    num_cameras = len(transforms.get("frames", []))
    print(f"✅ COLMAP completed in {elapsed:.1f}s — {num_cameras} camera poses estimated")
else:
    print("❌ COLMAP failed — transforms.json not found")
    print("   Try with more frames (increase FPS) or better video quality")

## Step 6: Train Gaussian Splatting Model\n\nTrains a Nerfstudio `splatfacto` model. This is the main GPU-intensive step.\n\n| Quality | Iterations | Time (T4 GPU) |\n|---|---|---|\n| Quick | 7,000 | ~5 min |\n| Standard | 15,000 | ~15 min |\n| High | 30,000 | ~30 min |

In [ ]:
print(f"🧠 Training splatfacto model ({QUALITY}: {ITERATIONS} iterations)...")
print(f"   Estimated time: {'~5 min' if QUALITY == 'quick' else '~15 min' if QUALITY == 'standard' else '~30 min'}")
start = time.time()

!ns-train splatfacto \
    --data "{PROCESSED_DIR}" \
    --output-dir "{MODEL_DIR}" \
    --max-num-iterations {ITERATIONS} \
    --steps-per-eval-image 500 \
    --pipeline.model.num-downscales 2 \
    --viewer.quit-on-train-completion True \
    2>&1 | grep -E "(Step|Loading|Saving|training|eta)" | tail -20

elapsed = time.time() - start
print(f"\n✅ Training completed in {elapsed/60:.1f} minutes")

## Step 7: Export to .splat File\n\nConverts the trained model to a `.splat` file for the web viewer.

In [ ]:
import glob

# Find the config file from training output
config_files = glob.glob(f"{MODEL_DIR}/**/config.yml", recursive=True)
if not config_files:
    print("❌ No config.yml found — training may have failed")
    print("   Check the training output above for errors")
else:
    config_path = config_files[0]
    print(f"📦 Exporting .splat from: {config_path}")

    !ns-export gaussian-splat \
        --load-config "{config_path}" \
        --output-dir "{OUTPUT_DIR}" \
        2>&1 | tail -5

    # Find the output file
    splat_files = glob.glob(f"{OUTPUT_DIR}/**/*.splat", recursive=True) + \
                  glob.glob(f"{OUTPUT_DIR}/**/*.ply", recursive=True)

    if splat_files:
        SPLAT_FILE = splat_files[0]
        size_mb = os.path.getsize(SPLAT_FILE) / 1024 / 1024
        print(f"✅ Exported: {SPLAT_FILE} ({size_mb:.1f} MB)")
    else:
        print("❌ Export failed — no .splat or .ply file found")

## Step 8: Download Your 3D Scene\n\nDownload the `.splat` file, then view it:\n\n1. Go to **https://captures-3d-ognvwkqc.devinapps.com**\n2. Click the upload icon (top right)\n3. Switch to **"Splat File"** tab\n4. Drag and drop your downloaded `.splat` file\n5. Explore your 3D scene! 🎉

In [ ]:
# Download the .splat file to your computer
from google.colab import files

if 'SPLAT_FILE' in dir() and os.path.exists(SPLAT_FILE):
    print(f"📥 Downloading {os.path.basename(SPLAT_FILE)}...")
    files.download(SPLAT_FILE)
    print(f"\n🎉 Done! Now view your 3D scene:")
    print(f"   1. Open https://captures-3d-ognvwkqc.devinapps.com")
    print(f"   2. Click upload icon → 'Splat File' tab")
    print(f"   3. Drop your .splat file")
else:
    print("❌ No .splat file to download. Check previous steps for errors.")

---\n\n## Troubleshooting\n\n| Problem | Solution |\n|---|---|\n| COLMAP fails | Use more frames (increase FPS to 3-5), ensure video has enough visual features |\n| Training is slow | Use "quick" quality for first test, upgrade to T4/A100 GPU |\n| Scene looks bad | Record slower, with more overlap between frames, in good lighting |\n| Out of memory | Reduce number of frames or use "quick" quality |\n| No GPU | Go to Runtime → Change runtime type → GPU (T4) |\n\n## Recording Tips for Best Results\n\n- **Speed**: Walk slowly, ~0.5 m/s\n- **Coverage**: 360° if possible, with lots of overlap\n- **Lighting**: Even, natural light is best\n- **Movement**: Smooth and steady — use a gimbal if available\n- **Avoid**: Fast motion, motion blur, reflective surfaces, transparent objects